In [2]:
from pipelines.readers.bcb_historico_taxas_juros import ReaderSQLBCBHistoricoTaxasJuros

from pandas import to_datetime, DataFrame, merge
from yfinance import download
import plotly.express as px
import plotly.graph_objects as go

In [3]:
# Ler os dados históricos de taxas de juros do Banco Central do Brasil (BCB) usando o leitor SQL.
reader = ReaderSQLBCBHistoricoTaxasJuros()
selic = reader.read(start_date=None, end_date=None)

print(selic.shape)
selic.tail(3)

(287, 10)


,Reunião,Data da Reunião,Data da Divulgação,Período de Vigência,Taxa Selic Meta (%),Taxa Selic Over (%),Taxa Selic Meta Anterior (%),Taxa Selic Over Anterior (%),Início da Vigência,Fim da Vigência
284,277ª,2026-03-18,NaT,19/03/2026 - 29/04/2026,14.75,NaN,1.53,14.65,2026-03-19,2026-04-29
285,278ª,2026-04-29,NaT,30/04/2026 - 17/06/2026,14.50,NaN,1.78,14.40,2026-04-30,2026-06-17
286,279ª,2026-06-17,NaT,18/06/2026 -,14.25,NaN,NaN,NaN,NaT,NaT


In [4]:
selic["Data da Reunião"] = to_datetime(selic["Data da Reunião"])
intervalo_medio = selic["Data da Reunião"].diff().mean()
intervalo_medio

Timedelta('38 days 06:42:47.832167832')

In [5]:
ativo = download("TOTS3.SA", period="max", interval="1d", progress=False, auto_adjust=False).droplevel(1, axis=1).reset_index()
ativo.tail(3)

Price,Date,Adj Close,Close,High,Low,Open,Volume
5053,2026-07-08,28.940001,28.940001,29.080000,28.209999,28.620001,3512400
5054,2026-07-09,29.590000,29.590000,29.690001,28.760000,29.000000,2652300
5055,2026-07-10,29.910000,29.910000,30.490000,29.700001,29.990000,4573200


In [6]:
selic_ativo = merge(selic, ativo, how='inner', left_on='Data da Reunião', right_on='Date')
selic_ativo.tail(3)

,Reunião,Data da Reunião,Data da Divulgação,Período de Vigência,Taxa Selic Meta (%),Taxa Selic Over (%),Taxa Selic Meta Anterior (%),Taxa Selic Over Anterior (%),Início da Vigência,Fim da Vigência,Date,Adj Close,Close,High,Low,Open,Volume
159,277ª,2026-03-18,NaT,19/03/2026 - 29/04/2026,14.75,NaN,1.53,14.65,2026-03-19,2026-04-29,2026-03-18,33.401733,33.779999,34.730000,33.669998,34.459999,5276800
160,278ª,2026-04-29,NaT,30/04/2026 - 17/06/2026,14.50,NaN,1.78,14.40,2026-04-30,2026-06-17,2026-04-29,31.009239,31.200001,31.480000,30.730000,31.379999,3585700
161,279ª,2026-06-17,NaT,18/06/2026 -,14.25,NaN,NaN,NaN,NaT,NaT,2026-06-17,27.930000,27.930000,29.040001,27.930000,29.000000,5715100


In [7]:
df = selic_ativo[["Date", "Taxa Selic Meta (%)", "Adj Close"]].copy()

df.set_index("Date", inplace=True)

df["Selic_variacao"] = df["Taxa Selic Meta (%)"].diff()
df["Retorno"] = df["Adj Close"].pct_change()

df = df.dropna()

df.tail(3)

,Taxa Selic Meta (%),Adj Close,Selic_variacao,Retorno
Date,,,,
2026-03-18,14.75,33.401733,-0.25,-0.286589
2026-04-29,14.50,31.009239,-0.25,-0.071628
2026-06-17,14.25,27.930000,-0.25,-0.099301


In [8]:
df["Selic_variacao"].corr(df["Retorno"])

np.float64(-0.1884239162435655)

In [9]:
rolling_corr = (
    df["Selic_variacao"]
    .rolling(window=12)
    .corr(df["Retorno"])
)

fig = px.line(
    rolling_corr,
    title="Correlação Rolling: Variação da Selic x Retorno",
    labels={
        "value": "Correlação",
        "Date": "Data"
    }
)

fig.add_hline(y=0)

fig.show()

In [10]:
fig = go.Figure()

interpretacao = rolling_corr.apply(
    lambda x: (
        "Correlação positiva: Selic e retorno tendem a se mover na mesma direção"
        if x > 0
        else "Correlação negativa: Selic e retorno tendem a se mover em direções opostas"
    )
)

# Preço histórico
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["Adj Close"],
        name="Adj Close",
        yaxis="y1"
    )
)

# Selic
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["Taxa Selic Meta (%)"],
        name="Selic",
        yaxis="y3"
    )
)

# Correlação rolling
fig.add_trace(
    go.Scatter(
        x=rolling_corr.index,
        y=rolling_corr,
        name="Correlação Rolling",
        yaxis="y2",
        hovertemplate=
            "Data: %{x}<br>"
            "Correlação: %{y:.2f}<br>"
            "%{text}<extra></extra>",
        text=interpretacao
    )
)

# Linha zero da correlação
fig.add_trace(
    go.Scatter(
        x=rolling_corr.index,
        y=[0] * len(rolling_corr),
        mode="lines",
        name="Zero",
        line=dict(dash="dash"),
        yaxis="y2"
    )
)

fig.update_layout(
    title="Preço x Selic x Correlação Rolling",

    # Eixo do preço
    yaxis=dict(
        title="Preço"
    ),

    # Eixo da correlação
    yaxis2=dict(
        title="Correlação",
        overlaying="y",
        side="right",
        range=[-1, 1]
    ),

    # Terceiro eixo (Selic)
    yaxis3=dict(
        title="Selic (%)",
        overlaying="y",
        side="left",
        position=0.05
    ),

    xaxis_title="Data",
    height=600
)

fig.show()